In [ ]:
import gzip
import json
import numpy as np
import os
import os.path as osp
import pandas as pd
import torch

from collections import defaultdict
from torch_geometric.data import extract_zip
from torch_geometric.data import HeteroData
from sentence_transformers import SentenceTransformer
from torch_geometric.io import fs
from gensim.models import FastText

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.8 MB/s eta 0:00:00


In [ ]:
def train_test_split(path,seq_len=15):
    users_id = []
    df_types = ['train', 'valid', 'test']
    seqs = {type: defaultdict(list) for type in df_types}
    with open(os.path.join(path, 'sequential_data.txt'), 'r') as file:
        for row in file:
            row_lst = list(map(int, row.strip().split()))
            users_id.append(row_lst[0])
            items = [i-1 for i in row_lst[1:]]
            items_types = {'train':items[:-2], 'valid': items[-(seq_len + 2):-2], 'test': items[-(seq_len+1):-1]}
            for tp in df_types:
                cur_items = items_types[tp]
                if tp != 'train':
                    gaps = [-1] * (seq_len - len(cur_items))
                    seqs[tp]['item_ID'].append(cur_items + gaps)
                else:
                    seqs[tp]['item_ID'].append(cur_items)
                seqs[tp]['item_ID_next'].append(items[-1 if tp =='test' else -2])
        for tp in df_types:
            seqs[tp]['user_ID'] = users_id
            seqs[tp] = pd.DataFrame(seqs[tp])
        return seqs



In [ ]:
path = r''
dfs = train_test_split(path)

In [ ]:
train, val, test = dfs['train'], dfs['valid'], dfs['test']

In [ ]:
train.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2]",3,1
1,"[5, 6, 7, 8, 9]",3,2
2,"[3, 11, 12, 13, 14, 15, 16]",17,3
3,"[19, 20, 21, 22]",3,4
4,"[3, 24, 25, 26, 27, 28, 29]",30,5
5,"[32, 33, 34, 3, 35, 36, 37, 38, 39, 40, 41, 42...",47,6
6,"[49, 50, 51, 52, 53]",54,7
7,"[55, 56, 57]",3,8
8,"[59, 60, 61, 21, 62, 63, 64, 65, 66, 67, 68, 6...",81,9
9,"[83, 84, 85, 82, 86, 58, 87, 88]",89,10


In [ ]:
val.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2, -1, -1, -1, -1, -1, -1, -1, -1, -1, ...",3,1
1,"[5, 6, 7, 8, 9, -1, -1, -1, -1, -1, -1, -1, -1...",3,2
2,"[3, 11, 12, 13, 14, 15, 16, -1, -1, -1, -1, -1...",17,3
3,"[19, 20, 21, 22, -1, -1, -1, -1, -1, -1, -1, -...",3,4
4,"[3, 24, 25, 26, 27, 28, 29, -1, -1, -1, -1, -1...",30,5
5,"[33, 34, 3, 35, 36, 37, 38, 39, 40, 41, 42, 43...",47,6
6,"[49, 50, 51, 52, 53, -1, -1, -1, -1, -1, -1, -...",54,7
7,"[55, 56, 57, -1, -1, -1, -1, -1, -1, -1, -1, -...",3,8
8,"[66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 7...",81,9
9,"[83, 84, 85, 82, 86, 58, 87, 88, -1, -1, -1, -...",89,10


In [ ]:
test.head(10)

,item_ID,item_ID_next,user_ID
0,"[0, 1, 2, 3, -1, -1, -1, -1, -1, -1, -1, -1, -...",4,1
1,"[5, 6, 7, 8, 9, 3, -1, -1, -1, -1, -1, -1, -1,...",10,2
2,"[3, 11, 12, 13, 14, 15, 16, 17, -1, -1, -1, -1...",18,3
3,"[19, 20, 21, 22, 3, -1, -1, -1, -1, -1, -1, -1...",23,4
4,"[3, 24, 25, 26, 27, 28, 29, 30, -1, -1, -1, -1...",31,5
5,"[34, 3, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44...",48,6
6,"[49, 50, 51, 52, 53, 54, -1, -1, -1, -1, -1, -...",3,7
7,"[55, 56, 57, 3, -1, -1, -1, -1, -1, -1, -1, -1...",58,8
8,"[67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 7...",82,9
9,"[83, 84, 85, 82, 86, 58, 87, 88, 89, -1, -1, -...",90,10


In [ ]:
train.columns

Index(['item_ID', 'item_ID_next', 'user_ID'], dtype='object')

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22363 entries, 0 to 22362
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   item_ID       22363 non-null  object
 1   item_ID_next  22363 non-null  int64 
 2   user_ID       22363 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 524.3+ KB


In [ ]:
train = train.iloc[:,[2,1,0]]
val = val.iloc[:, [2,1,0]]
test = test.iloc[:, [2,1,0]]
train

,user_ID,item_ID_next,item_ID
0,1,3,"[0, 1, 2]"
1,2,3,"[5, 6, 7, 8, 9]"
2,3,17,"[3, 11, 12, 13, 14, 15, 16]"
3,4,3,"[19, 20, 21, 22]"
4,5,30,"[3, 24, 25, 26, 27, 28, 29]"
...,...,...,...
22358,22359,11811,"[11793, 11810, 12041]"
22359,22360,3024,"[10746, 3022, 5594, 6465, 9743]"
22360,22361,11795,"[12054, 11802, 9267]"
22361,22362,3033,"[3022, 9743, 10606]"


In [ ]:
train.isnull().sum()

,0
user_ID,0
item_ID_next,0
item_ID,0


In [ ]:
val.isnull().sum()

,0
user_ID,0
item_ID_next,0
item_ID,0


In [ ]:
def df_to_dict_tensor(df, cols):
        res = {}
        for col in cols:
            if isinstance(df[col].iloc[0], list):
                if df[col].apply(len).nunique() == 1:
                    res[col] = torch.tensor(df[col].to_list(), dtype = torch.int64)
                else:
                    res[col] = df['item_ID'].to_list()
            else:
                res[col] = torch.tensor(df[col].to_numpy())
        for col in cols:
            next_col = col + '_next'
            if next_col in df.columns:
                res[next_col] = torch.tensor(df[next_col].to_numpy())
        res['user_ID'] = torch.from_numpy(df['user_ID'].to_numpy())
        return res

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield eval(l)

In [ ]:
path = '/content/'
df_graph = HeteroData()
with open(os.path.join(path,'datamaps.json'), 'r') as file:
    maps = json.load(file)
hist = {df_type: df_to_dict_tensor(ds, ['item_ID'])
        for df_type, ds in dfs.items()}
df_graph['user', 'rated', 'item'].history = hist
asin2id = pd.DataFrame([{"asin": key, "id": int(val) - 1} for key, val in maps["item2id"].items()])
item_data = pd.DataFrame([meta for meta in parse(path=os.path.join(path, "meta.json.gz"))]).merge(asin2id, on="asin").sort_values(by="id").fillna({"brand": "Unknown"})
sentences = item_data.apply(('id',
        lambda row:
            "Title: " +
            str(row["title"]) + "; " +
            "Brand: " +
            str(row["brand"]) + "; " +
            "Categories: " +
             str(row["categories"][0]) + "; " +
             "Price: " +
             str(row["price"]) + "; "),
             axis=1)


In [ ]:
item_data

,asin,description,title,imUrl,salesRank,categories,price,related,brand,id
6845,B004756YJA,OPI Burlesque Colors,"OPI Nail Lacquer, Simmer and Shimmer, 0.5-Flui...",http://ecx.images-amazon.com/images/I/411jo-OU...,{'Beauty': 46572},"[[Beauty, Makeup, Nails, Nail Polish]]",12.00,"{'also_bought': ['B0045M2T12', 'B004KFNHLA', '...",OPI,0
7871,B004ZT0SSG,Red Shatter Nail Polish\nFull Size :15ML,OPI Red Shatter Crackle Nail Polish E55 New,http://ecx.images-amazon.com/images/I/41X8hWnt...,{'Beauty': 74739},"[[Beauty, Makeup, Nails, Nail Polish]]",3.04,"{'also_bought': ['B004Y6G910', 'B005GSWUY0', '...",OPI,1
4584,B0020YLEYK,It is 3 effects function beblesh balm. By Aden...,SKIN79 The Prestige Beblesh Balm BB Cream Diam...,http://ecx.images-amazon.com/images/I/31lrzUjx...,{'Beauty': 24042},"[[Beauty, Skin Care, Face, Creams & Moisturize...",14.96,"{'also_bought': ['B006RWW7VU', 'B002HPBF32', '...",Unknown,2
0,7806397051,An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA,3
5405,B002WLWX82,Paraffin bath for pain relief and removing dry...,Dr. Scholl's Quick Heat Paraffin Spa Bath,http://ecx.images-amazon.com/images/I/41jOWVYU...,{'Beauty': 344},"[[Beauty, Skin Care, Hands & Nails, Paraffin B...",47.95,"{'also_bought': ['B000BLS0NM', 'B0006Q00IK', '...",Dr. Scholl&#39;s,4
...,...,...,...,...,...,...,...,...,...,...
11809,B00GYN9A08,One In Eight Ingredients In Personal Care Prod...,Best INDIAN HEALING CLAY -&quot;SODIUM&quot; B...,http://ecx.images-amazon.com/images/I/51JW3-Ie...,{'Beauty': 6825},"[[Beauty, Skin Care, Face, Treatments & Masks,...",25.99,"{'also_bought': ['B00JVXVBQE', 'B00HRGBSYW', '...",Unknown,12096
11937,B00IBMV2ME,The Best BOTANICAL HYALURONIC ACID (5.0%) Gel ...,Best Botanical Hyaluronic Acid Anti Aging Faci...,http://ecx.images-amazon.com/images/I/4171BmUV...,{'Beauty': 116649},"[[Beauty, Skin Care, Face, Oils & Serums]]",24.50,"{'also_bought': ['B00IC8JBIE', 'B00IC9AG5A', '...",Unknown,12097
11941,B00IC9AG5A,Announcing a Dermatologist Grade Skin Treatmen...,Anti Aging All In One Facial Treatment (Replac...,http://ecx.images-amazon.com/images/I/314b-jZn...,{'Beauty': 84262},"[[Beauty, Skin Care, Eyes, Combinations]]",26.50,"{'also_bought': ['B00IC8JBIE', 'B00IC7L3JK', '...",Unknown,12098
11965,B00IKKORVU,Announcing The Ultimate Vitamin C Anti Aging S...,Best Vitamin C Anti Aging 6 Item System &amp; ...,http://ecx.images-amazon.com/images/I/51yIcFHj...,{'Beauty': 87595},"[[Beauty, Skin Care, Sets & Kits]]",125.00,"{'also_viewed': ['B00IC8JBIE', 'B00GYJWL7G', '...",Unknown,12099


In [ ]:
sentences

,id,<lambda>
6845,0,"Title: OPI Nail Lacquer, Simmer and Shimmer, 0..."
7871,1,Title: OPI Red Shatter Crackle Nail Polish E55...
4584,2,Title: SKIN79 The Prestige Beblesh Balm BB Cre...
0,3,Title: WAWO 15 Color Professionl Makeup Eyesha...
5405,4,Title: Dr. Scholl's Quick Heat Paraffin Spa Ba...
...,...,...
11809,12096,Title: Best INDIAN HEALING CLAY -&quot;SODIUM&...
11937,12097,Title: Best Botanical Hyaluronic Acid Anti Agi...
11941,12098,Title: Anti Aging All In One Facial Treatment ...
11965,12099,Title: Best Vitamin C Anti Aging 6 Item System...


In [ ]:
tokenized_sentences = [
    s.lower().split()
    for s in sentences['<lambda>'].tolist()
]
model = FastText(
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

model.build_vocab(tokenized_sentences)
model.train(
    tokenized_sentences,
    total_examples=len(tokenized_sentences),
    epochs=100
)



(21160218, 29095200)

In [ ]:
def sentence_embedding(tokens, model):
    vecs = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)


In [ ]:
item_emb = np.vstack([
    sentence_embedding(tokens, model)
    for tokens in tokenized_sentences
])

item_emb = torch.tensor(item_emb, dtype=torch.float)


In [ ]:
item_data.head()

,asin,description,title,imUrl,salesRank,categories,price,related,brand,id
6845,B004756YJA,OPI Burlesque Colors,"OPI Nail Lacquer, Simmer and Shimmer, 0.5-Flui...",http://ecx.images-amazon.com/images/I/411jo-OU...,{'Beauty': 46572},"[[Beauty, Makeup, Nails, Nail Polish]]",12.00,"{'also_bought': ['B0045M2T12', 'B004KFNHLA', '...",OPI,0
7871,B004ZT0SSG,Red Shatter Nail Polish\nFull Size :15ML,OPI Red Shatter Crackle Nail Polish E55 New,http://ecx.images-amazon.com/images/I/41X8hWnt...,{'Beauty': 74739},"[[Beauty, Makeup, Nails, Nail Polish]]",3.04,"{'also_bought': ['B004Y6G910', 'B005GSWUY0', '...",OPI,1
4584,B0020YLEYK,It is 3 effects function beblesh balm. By Aden...,SKIN79 The Prestige Beblesh Balm BB Cream Diam...,http://ecx.images-amazon.com/images/I/31lrzUjx...,{'Beauty': 24042},"[[Beauty, Skin Care, Face, Creams & Moisturize...",14.96,"{'also_bought': ['B006RWW7VU', 'B002HPBF32', '...",Unknown,2
0,7806397051,An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA,3
5405,B002WLWX82,Paraffin bath for pain relief and removing dry...,Dr. Scholl's Quick Heat Paraffin Spa Bath,http://ecx.images-amazon.com/images/I/41jOWVYU...,{'Beauty': 344},"[[Beauty, Skin Care, Hands & Nails, Paraffin B...",47.95,"{'also_bought': ['B000BLS0NM', 'B0006Q00IK', '...",Dr. Scholl&#39;s,4


In [ ]:
def dict_tolist(dict_series):
  if isinstance(dict_series, float):
    return []
  if isinstance(dict_series, dict):
    all_info = []
    for cat in dict_series.values():
        all_info.extend(cat)
    return list(set(all_info))

def related_included(list_series):
  set_inp = set(list_series)
  unique = list(set_inp.intersection(asins))
  res = []
  for asin in unique:
    res.append(int(maps['item2id'][asin])-1)
  return res

In [ ]:
item_data['related_all'] = item_data['related'].apply(dict_tolist)
item_data['related_all'].head()

In [ ]:
asins = set(item_data['asin'].tolist())
item_data['related_included'] = item_data['related_all'].apply(related_included)
item_data['related_included'].head()

In [ ]:
item_data['related_included'].describe()

,related_included
count,12101
unique,11842
top,[]
freq,198


In [ ]:
item_data.head()

,asin,description,title,imUrl,salesRank,categories,price,related,brand,id,related_all,related_included
6845,B004756YJA,OPI Burlesque Colors,"OPI Nail Lacquer, Simmer and Shimmer, 0.5-Flui...",http://ecx.images-amazon.com/images/I/411jo-OU...,{'Beauty': 46572},"[[Beauty, Makeup, Nails, Nail Polish]]",12.00,"{'also_bought': ['B0045M2T12', 'B004KFNHLA', '...",OPI,0,"[B009ZHUYJ6, B0056K8RP4, B00BIY6D9O, B005R3H2H...","[1713, 9872, 3741, 6910, 9701, 3188, 2054, 490..."
7871,B004ZT0SSG,Red Shatter Nail Polish\nFull Size :15ML,OPI Red Shatter Crackle Nail Polish E55 New,http://ecx.images-amazon.com/images/I/41X8hWnt...,{'Beauty': 74739},"[[Beauty, Makeup, Nails, Nail Polish]]",3.04,"{'also_bought': ['B004Y6G910', 'B005GSWUY0', '...",OPI,1,"[B0056K8RP4, B004Y6G910, B002D4OJJE, B005A11VF...","[298, 147, 1713, 6289, 9872, 3741, 6910, 2054,..."
4584,B0020YLEYK,It is 3 effects function beblesh balm. By Aden...,SKIN79 The Prestige Beblesh Balm BB Cream Diam...,http://ecx.images-amazon.com/images/I/31lrzUjx...,{'Beauty': 24042},"[[Beauty, Skin Care, Face, Creams & Moisturize...",14.96,"{'also_bought': ['B006RWW7VU', 'B002HPBF32', '...",Unknown,2,"[B009JXNLX2, B0028M9SWO, B001CTXRNU, B004R6HIR...","[4583, 5829, 1724, 10292, 11152, 2132, 4454, 3..."
0,7806397051,An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA,3,"[B00DQ2ILQY, B004UO2GMW, 1111306923, B00IBVE79...","[3287, 30, 2171, 1474, 105, 31, 9111, 1582, 98..."
5405,B002WLWX82,Paraffin bath for pain relief and removing dry...,Dr. Scholl's Quick Heat Paraffin Spa Bath,http://ecx.images-amazon.com/images/I/41jOWVYU...,{'Beauty': 344},"[[Beauty, Skin Care, Hands & Nails, Paraffin B...",47.95,"{'also_bought': ['B000BLS0NM', 'B0006Q00IK', '...",Dr. Scholl&#39;s,4,"[B000E3CG00, B0002VALQK, B0001HA8VS, B005FLN2F...","[9014, 9013, 6193, 11838, 7421, 1484, 4375, 63..."


In [ ]:
sentences

,id,<lambda>
6845,0,"Title: OPI Nail Lacquer, Simmer and Shimmer, 0..."
7871,1,Title: OPI Red Shatter Crackle Nail Polish E55...
4584,2,Title: SKIN79 The Prestige Beblesh Balm BB Cre...
0,3,Title: WAWO 15 Color Professionl Makeup Eyesha...
5405,4,Title: Dr. Scholl's Quick Heat Paraffin Spa Ba...
...,...,...
11809,12096,Title: Best INDIAN HEALING CLAY -&quot;SODIUM&...
11937,12097,Title: Best Botanical Hyaluronic Acid Anti Agi...
11941,12098,Title: Anti Aging All In One Facial Treatment ...
11965,12099,Title: Best Vitamin C Anti Aging 6 Item System...


In [ ]:
df_graph['item'].x = item_emb
df_graph['item'].text = np.array(sentences)
df_graph['item'].related = item_data['related_included'].tolist()
gen = torch.Generator()
gen.manual_seed(42)
df_graph['item'].is_train = torch.rand(item_emb.shape[0], generator=gen) > 0.05

In [ ]:
item_emb

tensor([[ 0.4281, -0.2009, -0.1386,  ..., -0.4203, -0.0340,  0.2981],
        [ 0.5834, -0.1537, -0.3388,  ..., -0.6274, -0.0735,  0.2282],
        [ 0.5144,  0.3662, -0.4106,  ..., -0.3387, -0.0950,  0.0980],
        ...,
        [ 0.5821,  0.4641, -0.3705,  ..., -0.6340,  0.1260,  0.3243],
        [ 0.5615,  0.4495, -0.3056,  ..., -0.6608,  0.0316,  0.3455],
        [ 0.6409,  0.5084, -0.4816,  ..., -0.5172,  0.0383,  0.3217]])

In [ ]:
df_graph

HeteroData(
  item={
    x=[12101, 100],
    text=[12101, 2],
    related=[12101],
    is_train=[12101],
  },
  (user, rated, item)={
    history={
      train={
        item_ID=[22363],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      valid={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      test={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
    },
  }
)

In [ ]:
torch.save(df_graph, 'heterodata_object12_updated.pt')

In [ ]:
loaded_data = torch.load('heterodata_object12_updated.pt', weights_only=False)
loaded_data